# Penetration Test — OpenMRS Appointment Scheduling Module

**Auteur:** Liam / Martijn
**Datum:** 2026-06-18
**Target:** REST API + DWR layer
**Base URL:** `http://localhost/openmrs`

> **Doel:** Aantonen dat kwetsbaarheden bestonden vóór hardening (PR #97),
> en dat ze daarna daadwerkelijk gesloten zijn.
> Relevante tests worden twee keer uitgevoerd:
> op commit `bf6ab5b` **(vóór)** en op branch `develop` HEAD `b6364c4` **(ná)**.

## Voorbereiding

Start de lokale OpenMRS stack:

```bash
docker compose -f docker-compose.yml -f docker-compose.dev.yml up -d
```

OpenMRS UI: **http://localhost/openmrs** — `admin / Admin123`

> **Let op:** OpenMRS kent geen directe privilege-toewijzing aan gebruikers — je wijst *rollen* toe.
> Privileges worden door de module geregistreerd in `config.xml` en zijn zichtbaar bij het aanmaken van een rol.
> Zorg dat de appointment-module actief is (zie Module health check cel) vóórdat je de rol aanmaakt.

### Stap 1 — Maak een custom rol aan

**Administration → Manage Roles → Add Role**

| Veld | Waarde |
|------|--------|
| Role | `Appointment Viewer` |
| Description | Low-privilege test role for pentest |
| Privileges | Vink aan: **View Appointments** |

Sla op. Zorg dat je **niet** `View Appointments Statistics` of `Schedule Appointments` aanvinkt.

### Stap 2 — Maak de testgebruiker aan

**Administration → Manage Users → Add User**

| Veld | Waarde |
|------|--------|
| Given name | Nurse |
| Family name | Test |
| Gender | Female |
| Username | `nurse_test` |
| Password | `Nurse123` |
| Roles | Vink aan: **Appointment Viewer** (de rol die je net maakte) |

> `Anonymous` en `Authenticated` worden automatisch toegewezen — niet nodig om die aan te vinken.

### Stap 3 — Verifieer de setup

Voer cel 0 (Setup) hieronder uit. De cel controleert of beide sessies succesvol authenticeren voordat je verdergaat.

## 0 — Setup & hulpfuncties

In [3]:
import requests, json, sys

BASE_URL  = "http://localhost/openmrs"
REST_BASE = f"{BASE_URL}/ws/rest/v1/appointmentscheduling"

ADMIN_USER    = "admin"
ADMIN_PASS    = "Admin123"
LOW_PRIV_USER = "nurse_test"
LOW_PRIV_PASS = "Nurse123"

def make_session(username, password):
    s = requests.Session()
    s.auth = (username, password)
    s.headers.update({"Content-Type": "application/json", "Accept": "application/json"})
    return s

admin_session    = make_session(ADMIN_USER, ADMIN_PASS)
low_priv_session = make_session(LOW_PRIV_USER, LOW_PRIV_PASS)
anon_session     = requests.Session()

def pp(label, resp):
    print(f"[{label}] HTTP {resp.status_code}")
    try:
        print(json.dumps(resp.json(), indent=2)[:600])
    except Exception:
        print(resp.text[:400])
    print()

# --- Session verificatie ---
print("=== Session verificatie ===")
_auth_ep = f"{BASE_URL}/ws/rest/v1/session"
_ok = True

r_admin_check = admin_session.get(_auth_ep)
if r_admin_check.status_code == 200 and r_admin_check.json().get("authenticated"):
    print(f"[OK]   admin ({ADMIN_USER}) — geauthenticeerd")
else:
    print(f"[FOUT] admin ({ADMIN_USER}) — HTTP {r_admin_check.status_code}, niet geauthenticeerd")
    _ok = False

r_nurse_check = low_priv_session.get(_auth_ep)
if r_nurse_check.status_code == 200 and r_nurse_check.json().get("authenticated"):
    print(f"[OK]   low-priv ({LOW_PRIV_USER}) — geauthenticeerd")
else:
    print(f"[FOUT] low-priv ({LOW_PRIV_USER}) — HTTP {r_nurse_check.status_code}, niet geauthenticeerd")
    print("       Controleer: gebruiker bestaat? wachtwoord klopt? rol 'Appointment Viewer' toegewezen?")
    _ok = False

if not _ok:
    print("\nSetup onvolledig — los bovenstaande fouten op voor je verdergaat.")
else:
    print("\nSetup compleet — beide sessies geauthenticeerd.")

=== Session verificatie ===
[OK]   admin (admin) — geauthenticeerd
[OK]   low-priv (nurse_test) — geauthenticeerd

Setup compleet — beide sessies geauthenticeerd.


## 1 — Module health check

Als dit 404 geeft is de module niet actief.
Verwijder dan de Bahmni Appointment Scheduling module via de UI
en upload de eigen `.omod` via **Administration → Manage Modules → Add or Upgrade Module**.

In [4]:
resp = admin_session.get(f"{BASE_URL}/ws/rest/v1/module", params={"v": "default"})
if resp.status_code == 200:
    apt = [m for m in resp.json().get("results", [])
           if "appointment" in m.get("moduleId", "").lower()]
    if apt:
        for m in apt:
            print(f"  {m.get('moduleId')} v{m.get('version')} — gestart: {m.get('started')}")
    else:
        print("Geen appointment-module gevonden. Deploy de .omod eerst.")
else:
    print(f"HTTP {resp.status_code} — modules endpoint niet bereikbaar")

Geen appointment-module gevonden. Deploy de .omod eerst.


## 2 — Basis authenticatie (geen voor/na — altijd van kracht)

OpenMRS REST vereist altijd authenticatie via het platform zelf.
Dit is geen gevolg van PR #97 — het is een baseline controle.

De test werkt in twee delen:
1. **Core endpoints** (altijd aanwezig, ongeacht de module) — bewijst dat OpenMRS auth werkt.
2. **Module endpoints** — geeft 404 voor iedereen als de module nog niet geladen is; dat is verwacht gedrag, geen kwetsbaarheid. Laad de module en voer deze cel opnieuw uit om de volledige tabel te zien.

Verwacht op core endpoints: anoniem → 401/403, admin → 200/204/400.

In [7]:
# Deel 1: core OpenMRS endpoints — altijd aanwezig, bewijst dat auth werkt ongeacht de module
# /patient vereist ?q= param en geeft 400 voor iedereen zonder — slechte auth-testcandidate.
core_endpoints = [
    (f"{BASE_URL}/ws/rest/v1/concept",  "concept"),
    (f"{BASE_URL}/ws/rest/v1/session",  "session"),
]

print("=== Core endpoints (module-onafhankelijk) ===")
print(f"{'Endpoint':<40} {'Anon':<12} {'Admin':<12} {'Oordeel'}")
print("-" * 78)
for ep, name in core_endpoints:
    r_anon  = anon_session.get(ep)
    r_admin = admin_session.get(ep)
    if name == "session":
        anon_ok  = r_anon.status_code == 200 and not r_anon.json().get("authenticated", True)
        admin_ok = r_admin.status_code == 200 and r_admin.json().get("authenticated", False)
        verdict  = "OK" if anon_ok and admin_ok else "AFWIJKING"
        anon_label  = f"{r_anon.status_code}(unauth)" if anon_ok  else f"{r_anon.status_code}(!)"
        admin_label = f"{r_admin.status_code}(auth)"  if admin_ok else f"{r_admin.status_code}(!)"
        print(f"{ep:<40} {anon_label:<12} {admin_label:<12} {verdict}")
    else:
        verdict = "OK" if r_anon.status_code in (401, 403) and r_admin.status_code in (200, 204, 400) else "AFWIJKING"
        print(f"{ep:<40} {r_anon.status_code:<12} {r_admin.status_code:<12} {verdict}")

# Deel 2: module endpoints — werken alleen als de module geladen is
module_endpoints = [
    f"{REST_BASE}/appointment",
    f"{REST_BASE}/appointmentrequest",
    f"{REST_BASE}/timeslot",
    f"{REST_BASE}/appointmentblock",
    f"{REST_BASE}/appointmenttype",
    f"{REST_BASE}/providerschedule",
    f"{REST_BASE}/createappointment",
    f"{REST_BASE}/dailyappointmentcount",
]

print()
print("=== Module endpoints (vereisen geladen module) ===")
_probe       = anon_session.get(module_endpoints[0])
_admin_probe = admin_session.get(module_endpoints[0])
if _probe.status_code == 404 and _admin_probe.status_code == 404:
    print("Module nog niet geladen — alle endpoints geven 404 voor iedereen.")
    print("Dit is verwacht. Laad de .omod en voer deze cel opnieuw uit.")
else:
    print(f"{'Endpoint':<55} {'Anon':<8} {'Admin':<8} {'Oordeel'}")
    print("-" * 90)
    for ep in module_endpoints:
        r_anon  = anon_session.get(ep)
        r_admin = admin_session.get(ep)
        verdict = "OK" if r_anon.status_code in (401, 403) and r_admin.status_code in (200, 204, 400) else "AFWIJKING"
        print(f"{ep:<55} {r_anon.status_code:<8} {r_admin.status_code:<8} {verdict}")

=== Core endpoints (module-onafhankelijk) ===
Endpoint                                 Anon         Admin        Oordeel
------------------------------------------------------------------------------
http://localhost/openmrs/ws/rest/v1/concept 401          200          OK
http://localhost/openmrs/ws/rest/v1/session 200(unauth)  200(auth)    OK

=== Module endpoints (vereisen geladen module) ===
Module nog niet geladen — alle endpoints geven 404 voor iedereen.
Dit is verwacht. Laad de .omod en voer deze cel opnieuw uit.


---
---

# FASE 1 — VÓÓR HARDENING &nbsp; `bf6ab5b`

**Stap: switch naar pre-hardening commit en bouwen**

```bash
git checkout bf6ab5b
mvn clean package -DskipTests -q
```

`clean` is verplicht: zonder het worden `.class`-bestanden van de andere commit hergebruikt,
waardoor de `@Authorized`-annotaties mogelijk niet correct in de `.omod` terechtkomen.

Daarna in OpenMRS UI:
1. **Administration → Manage Modules** → `appointmentscheduling` → **Stop** → **Uninstall**
2. **Add or Upgrade Module** → upload `omod/target/appointmentscheduling-*.omod`
3. Wacht tot de module groen is, dan de cel hieronder uitvoeren.

In [ ]:
input("Druk Enter als pre-hardening build (bf6ab5b) actief is in OpenMRS... ")
print("Start VOOR-tests.")

### VOOR-3 — Privilege escalation: statistieken opvragen

**Verwacht: HTTP 200** — `AppointmentDailyCountController` had vóór PR #97 **geen** `@Authorized`.
Elke ingelogde gebruiker kon dagelijkse tellingen opvragen zonder statistiekenrecht.

```java
// AppointmentDailyCountController.java @ bf6ab5b
@Controller
@RequestMapping(".../dailyappointmentcount")
public class AppointmentDailyCountController {
    @RequestMapping(method = RequestMethod.GET)
    public @ResponseBody List<AppointmentDailyCount> getDailyAggregates(...) { ... }
    // geen @Authorized aanwezig
}
```

In [ ]:
url    = f"{REST_BASE}/dailyappointmentcount"
params = {"fromDate": "2026-01-01", "toDate": "2026-12-31"}

r_low   = low_priv_session.get(url, params=params)
r_admin = admin_session.get(url, params=params)

pp("low-priv  [VOOR]", r_low)
pp("admin     [VOOR]", r_admin)

verdict = "KWETSBAARHEID BEVESTIGD" if r_low.status_code not in (401, 403) else "onverwacht geblokkeerd"
print(f"low-priv HTTP {r_low.status_code} — {verdict}")

### VOOR-4 — Privilege escalation: afspraak aanmaken

**Verwacht: HTTP 400/500 (niet 403)** — `AppointmentRequisitionController` had vóór PR #97 **geen** `@Authorized`.
Elke ingelogde gebruiker kon POST-requests sturen naar het createappointment-endpoint.

```java
// AppointmentRequisitionController.java @ bf6ab5b
@Controller
@RequestMapping(".../createappointment")
public class AppointmentRequisitionController {
    @RequestMapping(method = RequestMethod.POST)
    @ResponseBody
    public AppointmentRequisition createAppointment(...) { ... }
    // geen @Authorized aanwezig
}
```

In [ ]:
url = f"{REST_BASE}/createappointment"

# Lege body — we testen bereikbaarheid van het endpoint, niet een succesvolle boeking.
# 400/500 bevestigt dat het endpoint bereikbaar is voor low-priv (had 403 moeten zijn).
r_low   = low_priv_session.post(url, json={})
r_admin = admin_session.post(url, json={})

pp("low-priv  [VOOR]", r_low)
pp("admin     [VOOR]", r_admin)

verdict = "KWETSBAARHEID BEVESTIGD (endpoint bereikbaar)" if r_low.status_code not in (401, 403) else "onverwacht geblokkeerd"
print(f"low-priv HTTP {r_low.status_code} — {verdict}")

### VOOR-7/8 — DWR: patiëntdata zonder privilege check

**Verwacht: patiëntdata teruggestuurd zonder role-check** —
`DWRAppointmentService.getPatientDescription()` had vóór PR #97 geen `requirePrivilege()`.
Elke ingelogde gebruiker ontving PHI: naam, telefoonnummer, gemiste afspraken.

```java
// DWRAppointmentService.java @ bf6ab5b
public PatientData getPatientDescription(Integer patientId) {
    Patient patient = Context.getPatientService().getPatient(patientId);  // geen check
    ...
    return patientData;  // PHI teruggegeven aan elke ingelogde gebruiker
}
```

In [ ]:
DWR_URL = f"{BASE_URL}/module/appointmentscheduling/dwr/call/plaincall/"

def dwr_call(session, method_name, params_dict):
    lines = [
        "callCount=1", "page=/openmrs/", "httpSessionId=", "scriptSessionId=ABC123",
        "c0-scriptName=DWRAppointmentService", f"c0-methodName={method_name}", "c0-id=0",
    ]
    for k, v in params_dict.items():
        lines.append(f"{k}={v}")
    lines.append("batchId=0")
    body = "\n".join(lines) + "\n"
    return session.post(DWR_URL, data=body, headers={"Content-Type": "text/plain"})

r_anon  = dwr_call(anon_session,     "getPatientDescription", {"c0-param0": "number:1"})
r_low   = dwr_call(low_priv_session, "getPatientDescription", {"c0-param0": "number:1"})
r_admin = dwr_call(admin_session,    "getPatientDescription", {"c0-param0": "number:1"})

print(f"Anoniem  [VOOR]: HTTP {r_anon.status_code}")
print(r_anon.text[:250])
print()
print(f"Low-priv [VOOR]: HTTP {r_low.status_code}")
print(r_low.text[:350])
print()
print(f"Admin    [VOOR]: HTTP {r_admin.status_code}")
print(r_admin.text[:350])
print()

phi_leaked = "fullName" in r_low.text or (r_low.status_code == 200 and "exception" not in r_low.text.lower())
print(f"PHI gelekt naar low-priv: {'KWETSBAARHEID BEVESTIGD' if phi_leaked else 'controleer output handmatig'}")

---
---

# FASE 2 — NÁ HARDENING &nbsp; `develop` (`b6364c4`)

**Stap: switch terug naar develop en bouwen**

```bash
git checkout develop
mvn clean package -DskipTests -q
```

Daarna in OpenMRS UI:
1. **Administration → Manage Modules** → `appointmentscheduling` → **Stop** → **Uninstall**
2. **Add or Upgrade Module** → upload `omod/target/appointmentscheduling-*.omod`
3. Wacht tot de module groen is, dan de cel hieronder uitvoeren.

In [ ]:
input("Druk Enter als post-hardening build (develop) actief is in OpenMRS... ")
print("Start NA-tests.")

### NA-3 — Privilege escalation: statistieken opvragen

**Verwacht: HTTP 403** —
`AppointmentDailyCountController` heeft nu `@Authorized(PRIV_VIEW_APPOINTMENTS_STATISTICS)`
op zowel klasse- als methode-niveau (Defense in Depth / Gatekeeper Pattern, PR #97).

```java
// AppointmentDailyCountController.java @ develop
@Controller
@Authorized(AppointmentUtils.PRIV_VIEW_APPOINTMENTS_STATISTICS)  // ← toegevoegd PR #97
public class AppointmentDailyCountController {
    @RequestMapping(method = RequestMethod.GET)
    @Authorized(AppointmentUtils.PRIV_VIEW_APPOINTMENTS_STATISTICS)  // ← defense in depth
    public List<AppointmentDailyCount> getDailyAggregates(...) { ... }
}
```

In [ ]:
url    = f"{REST_BASE}/dailyappointmentcount"
params = {"fromDate": "2026-01-01", "toDate": "2026-12-31"}

r_low   = low_priv_session.get(url, params=params)
r_admin = admin_session.get(url, params=params)

pp("low-priv  [NA]", r_low)
pp("admin     [NA]", r_admin)

if r_low.status_code in (401, 403):
    print(f"MITIGATIE EFFECTIEF: low-priv geblokkeerd (HTTP {r_low.status_code})")
else:
    print(f"MITIGATIE FAALT: low-priv krijgt HTTP {r_low.status_code}")

### NA-4 — Privilege escalation: afspraak aanmaken

**Verwacht: HTTP 403** —
`AppointmentRequisitionController` heeft nu `@Authorized(PRIV_SCHEDULE_APPOINTMENTS)`
op zowel klasse- als methode-niveau (PR #97).

```java
// AppointmentRequisitionController.java @ develop
@Controller
@Authorized(AppointmentUtils.PRIV_SCHEDULE_APPOINTMENTS)  // ← toegevoegd PR #97
public class AppointmentRequisitionController {
    @RequestMapping(method = RequestMethod.POST)
    @Authorized(AppointmentUtils.PRIV_SCHEDULE_APPOINTMENTS)  // ← defense in depth
    public AppointmentRequisition createAppointment(...) { ... }
}
```

In [ ]:
url = f"{REST_BASE}/createappointment"

r_low   = low_priv_session.post(url, json={})
r_admin = admin_session.post(url, json={})

pp("low-priv  [NA]", r_low)
pp("admin     [NA]", r_admin)

if r_low.status_code in (401, 403):
    print(f"MITIGATIE EFFECTIEF: low-priv geblokkeerd (HTTP {r_low.status_code})")
else:
    print(f"MITIGATIE FAALT: low-priv krijgt HTTP {r_low.status_code}")

### NA-7/8 — DWR: patiëntdata met privilege check

**Verwacht:** anoniem → geweigerd | low-priv → `APIAuthenticationException` | admin → data

```java
// DWRAppointmentService.java @ develop
public PatientData getPatientDescription(Integer patientId) {
    Context.requirePrivilege(AppointmentUtils.PRIV_VIEW_APPOINTMENTS);  // ← toegevoegd PR #97
    Patient patient = Context.getPatientService().getPatient(patientId);
    ...
}
```

> `dwr_call` is gedefinieerd in cel VOOR-7/8 — zorg dat die cel is uitgevoerd in deze kernel-sessie.

In [ ]:
r_anon  = dwr_call(anon_session,     "getPatientDescription", {"c0-param0": "number:1"})
r_low   = dwr_call(low_priv_session, "getPatientDescription", {"c0-param0": "number:1"})
r_admin = dwr_call(admin_session,    "getPatientDescription", {"c0-param0": "number:1"})

print(f"Anoniem  [NA]: HTTP {r_anon.status_code}")
print(r_anon.text[:250])
print()
print(f"Low-priv [NA]: HTTP {r_low.status_code}")
print(r_low.text[:350])
print()
print(f"Admin    [NA]: HTTP {r_admin.status_code}")
print(r_admin.text[:350])
print()

anon_blocked = r_anon.status_code != 200 or "exception" in r_anon.text.lower()
low_blocked  = "exception" in r_low.text.lower() or r_low.status_code not in (200,)
admin_ok     = "fullName" in r_admin.text or r_admin.status_code == 200

print(f"Anoniem geweigerd:  {'JA' if anon_blocked else 'NEE — FOUT'}")
print(f"Low-priv geweigerd: {'JA — MITIGATIE EFFECTIEF' if low_blocked else 'NEE — MITIGATIE FAALT'}")
print(f"Admin krijgt data:  {'JA' if admin_ok else 'NEE — controleer output'}")

---
---

## Overige tests — eenmalig op `develop` (geen voor/na)

De volgende kwetsbaarheden zijn **niet opgelost** binnen scope
of zijn niet geraakt door de hardening in PR #97.

### 5 — PHI exposure in GET URL (RI-12, open bevinding)

`GET /appointment?patient=<uuid>` zet een patiënt-UUID in de querystring.
Wordt vastgelegd in Tomcat access logs, proxy logs en browser history — NEN-7510 §5.14.
**Status: OPEN — geaccepteerde technical debt** (audit-report.md §5.14).

In [ ]:
resp = admin_session.get(f"{BASE_URL}/ws/rest/v1/patient",
                          params={"q": "Test", "v": "ref", "limit": 1})
patient_uuid = None
if resp.status_code == 200 and resp.json().get("results"):
    patient_uuid = resp.json()["results"][0]["uuid"]
    print(f"Test patient UUID: {patient_uuid}")
else:
    print(f"Geen testpatient gevonden via 'Test' — zoek handmatig een UUID in de OpenMRS UI")
    print(f"HTTP {resp.status_code}")

In [ ]:
if patient_uuid:
    url = f"{REST_BASE}/appointment?patient={patient_uuid}&v=default"
    print(f"Aanvraag-URL (PHI in querystring):")
    print(f"  {url}")
    print()
    r = admin_session.get(url)
    print(f"HTTP {r.status_code} — {len(r.json().get('results', []))} afspraken")
    print()
    print("BEVINDING RI-12 BEVESTIGD:")
    print("  Patient-UUID zichtbaar in URL-querystring.")
    print("  Wordt vastgelegd in Tomcat access log, proxy logs en browser history.")
    print("  Status: OPEN — geaccepteerde technical debt.")
    print("  Aanbeveling: migreer patientszoekacties van GET naar POST.")
else:
    print("Sla over — geen patient_uuid beschikbaar.")

### 6 — Open redirect (RI-18, GitHub #99, open bevinding)

`AppointmentBlockFormController.java:313` redirect na POST naar de waarde
van `redirectedFrom` parameter zonder validatie — phishing-vector.

```java
return "redirect:" + redirectedFrom;  // geen validatie van de target URL
```

Frontend geeft HTTP 404 wegens Bahmni-module conflict.
Test via Burp Suite zodra de module correct laadt, of verwijs naar broncode als statisch bewijs.
**Status: OPEN** (audit-report.md, GitHub #99).

In [ ]:
form_url = f"{BASE_URL}/openmrs/module/appointmentscheduling/appointmentBlockForm"
r_get    = admin_session.get(form_url)
print(f"GET form pagina: HTTP {r_get.status_code}")
print()

if r_get.status_code == 404:
    print("Frontend niet bereikbaar (404 — Bahmni module conflict).")
    print("Statisch bewijs: AppointmentBlockFormController.java:313")
    print('  return "redirect:" + redirectedFrom;  // geen URL-validatie')
    print()
    print("Open redirect aantoonbaar aanwezig in broncode, niet testbaar via HTTP.")
else:
    malicious = "https://evil.example.com/fake-openmrs-login"
    r_post = admin_session.post(
        form_url,
        data={"redirectedFrom": malicious, "emptyTypes": "no"},
        allow_redirects=False
    )
    print(f"POST met redirect naar: {malicious}")
    print(f"HTTP {r_post.status_code}")
    print(f"Location header: {r_post.headers.get('Location', 'geen')}")
    if "evil.example.com" in r_post.headers.get("Location", ""):
        print("BEVINDING #99 BEVESTIGD: open redirect naar extern domein")

### 9 — SQL/HQL injection probes

OpenMRS gebruikt Hibernate met geparametriseerde queries — directe SQL injection
is onwaarschijnlijk. Test op gedragsafwijkingen (HTTP 500) bij injectie-payloads.
**Verwacht: HTTP 400** (ongeldige UUID-waarde) of lege lijst — nooit HTTP 500.

In [ ]:
payloads = [
    "' OR '1'='1",
    "'; DROP TABLE appointment; --",
    "1 AND 1=1",
    "%27%20OR%20%271%27%3D%271",
]

print(f"{'Payload':<40} {'Status':<8} {'Oordeel'}")
print("-" * 68)
for p in payloads:
    r = admin_session.get(f"{REST_BASE}/appointment", params={"patient": p})
    ok = "OK" if r.status_code in (200, 400) else f"AFWIJKING"
    print(f"{p:<40} {r.status_code:<8} {ok}")

---

## Eindoverzicht — voor/na vergelijking

Vervang de `???` placeholders met de daadwerkelijke HTTP-statuscodes na het uitvoeren van alle tests.

In [ ]:
voor_na = [
    # (id, omschrijving, voor_http, voor_oordeel, na_http, na_oordeel)
    ("T-03", "Low-priv stats opvragen",      "???", "KWETSBAAR / OK", "???", "GEMITIGEERD / FAALT"),
    ("T-04", "Low-priv afspraak aanmaken",   "???", "KWETSBAAR / OK", "???", "GEMITIGEERD / FAALT"),
    ("T-07", "DWR anoniem getPatientDesc",   "???", "KWETSBAAR / OK", "???", "GEMITIGEERD / FAALT"),
    ("T-08", "DWR low-priv getPatientDesc",  "???", "KWETSBAAR / OK", "???", "GEMITIGEERD / FAALT"),
]

altijd = [
    # (id, omschrijving, http, oordeel)
    ("T-01", "Anon. access geblokkeerd (alle endpoints)", "???", "OK / FAALT"),
    ("T-05", "PHI in GET URL (RI-12)",                    "200", "OPEN — geaccepteerd"),
    ("T-06", "Open redirect (#99)",                       "404", "OPEN — niet testbaar via HTTP"),
    ("T-09", "SQL injection probes",                      "???", "OK / FAALT"),
]

print("VOOR / NA VERGELIJKING:")
print(f"{'ID':<6} {'Test':<42} {'VOOR':<8} {'VOOR oordeel':<26} {'NA':<8} {'NA oordeel'}")
print("-" * 105)
for t in voor_na:
    print(f"{t[0]:<6} {t[1]:<42} {t[2]:<8} {t[3]:<26} {t[4]:<8} {t[5]}")

print()
print("ALTIJD GELDIG (geen voor/na):")
print(f"{'ID':<6} {'Test':<52} {'HTTP':<8} {'Oordeel'}")
print("-" * 82)
for t in altijd:
    print(f"{t[0]:<6} {t[1]:<52} {t[2]:<8} {t[3]}")